# Silver Brasil — Espelho governado da Bronze

## Objetivo

Aplicar tipagem, padronização e governança aos dados
de chegadas de turistas internacionais ao Brasil.

Fonte: Ministério do Turismo
Período: 1990–2025

Regras:
- Preservar todos os registros da Bronze.
- Não aplicar filtros ou agregações de negócio.
- Não eliminar registros duplicados nesta etapa.
- Manter a rastreabilidade dos arquivos de origem.
- Validar os dados antes de materializar a Silver.

In [0]:

from pyspark.sql import functions as F

TABELA_BRONZE = "chegadas.brasil.bronze"
TABELA_SILVER = "chegadas.silver.brasil"

df_bronze = spark.table(TABELA_BRONZE)

Inspeção de qualidade e tipagem

In [0]:
df_bronze.printSchema()

In [0]:
print("Total de registros:", df_bronze.count())
print("Total de colunas:", len(df_bronze.columns))

In [0]:
display(
    df_bronze
    .groupBy("ano")
    .count()
    .orderBy(F.col("ano").desc())
)

In [0]:

diagnostico_null = df_bronze.select([
    F.sum(
        F.when(F.col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in df_bronze.columns
])

display(diagnostico_null)

In [0]:
display(
    df_bronze
    .filter(F.col("chegadas").isNull())
    .groupBy("_arquivo_origem")
    .count()
    .orderBy("_arquivo_origem")
)

Entendendo observações: 1) caso dos regristros sem chegadas.

In [0]:
display(
    df_bronze
    .filter(F.col("chegadas").isNull())
    .select(
        "continente",
        "pais",
        "uf",
        "via",
        "ano",
        "mes",
        "chegadas",
        "_arquivo_origem"
    )
    .limit(20)
)

In [0]:
display(
    df_bronze
    .filter(F.col("chegadas").isNull())
    .groupBy(
        "_arquivo_origem",
        "mes"
    )
    .count()
    .orderBy(
        "_arquivo_origem",
        F.desc("count")
    )
)

In [0]:
display(
    df_bronze
    .filter(F.col("chegadas").isNull())
    .groupBy("ano", "pais")
    .count()
    .orderBy("ano", F.desc("count"))
)

In [0]:
display(
    df_bronze
    .filter(F.col("chegadas").isNull())
    .groupBy(
        "ano",
        "uf",
        "via"
    )
    .count()
    .orderBy(
        "ano",
        "uf",
        "via"
    )
)

Criação e checagem da Tabela Silver

In [0]:
spark.sql("""CREATE SCHEMA IF NOT EXISTS chegadas.silver""")

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE chegadas.silver.brasil AS

SELECT
    continente,
    cod_continente,
    pais,
    cod_pais,
    uf,
    cod_uf,
    via,
    cod_via,
    try_cast(ano AS INT) AS ano,
    mes,
    cod_mes,
    try_cast(chegadas AS BIGINT) AS chegadas,
    _arquivo_origem,
    ingerido_em,
    current_timestamp() AS _transformado_em

FROM chegadas.brasil.bronze
""")

print("Tabela Silver Brasil criada!")

In [0]:
display(spark.sql("""
SELECT
    (SELECT COUNT(*)
     FROM chegadas.brasil.bronze) AS bronze,

    (SELECT COUNT(*)
     FROM chegadas.silver.brasil) AS silver,

    (SELECT COUNT(*)
     FROM chegadas.brasil.bronze)
    -
    (SELECT COUNT(*)
     FROM chegadas.silver.brasil) AS diferenca
"""))

In [0]:

df_silver = spark.table("chegadas.silver.brasil")

df_silver.printSchema()

In [0]:
display(spark.sql("""
SELECT
    COUNT(*) AS total_registros,

    COUNT(chegadas) AS chegadas_preenchidas,

    COUNT(*) - COUNT(chegadas) AS chegadas_nulas,

    COUNT(ano) AS anos_preenchidos

FROM chegadas.silver.brasil
"""))

Documentação

In [0]:

COMENTARIOS_BRASIL = {
    "continente": "Continente de origem do visitante, conforme a classificação do Ministério do Turismo. Ausente no arquivo de 2025.",
    "cod_continente": "Código do continente de origem informado pela fonte. Ausente no arquivo de 2025.",
    "pais": "País de origem do visitante, conforme informado pelo Ministério do Turismo.",
    "cod_pais": "Código do país de origem informado pela fonte. Ausente no arquivo de 2025.",
    "uf": "Unidade federativa de entrada do visitante no Brasil. Não representa necessariamente o destino turístico visitado.",
    "cod_uf": "Código da unidade federativa de entrada informado pela fonte. Ausente no arquivo de 2025.",
    "via": "Via de acesso utilizada para a entrada no Brasil, como aérea, terrestre, marítima ou fluvial.",
    "cod_via": "Código da via de acesso informado pela fonte. Ausente no arquivo de 2025.",
    "ano": "Ano de referência do registro de chegada internacional.",
    "mes": "Mês de referência do registro de chegada internacional.",
    "cod_mes": "Código do mês informado pela fonte. Ausente no arquivo de 2025.",
    "chegadas": "Quantidade de chegadas internacionais registrada para a combinação de país, UF, via de acesso, ano e mês. Valores nulos representam informação ausente, não zero chegadas.",
    "_arquivo_origem": "Nome do arquivo CSV original do qual o registro foi ingerido.",
    "ingerido_em": "Data e horário em que o registro foi ingerido na camada Bronze.",
    "_transformado_em": "Data e horário em que a tabela Silver foi construída a partir da Bronze."
}

for coluna, comentario in COMENTARIOS_BRASIL.items():
    spark.sql(
        f"ALTER TABLE chegadas.silver.brasil "
        f"ALTER COLUMN `{coluna}` COMMENT '{comentario}'"
    )

print(f"{len(COMENTARIOS_BRASIL)} colunas documentadas.")

In [0]:
spark.sql("""
COMMENT ON TABLE chegadas.silver.brasil IS
'Silver - Chegadas de turistas internacionais ao Brasil,
1990 a 2025. Espelho governado da Bronze, sem exclusão
de registros ou agregações. Ano e chegadas convertidos
para tipos numéricos. Valores ausentes preservados.
Fonte: Ministério do Turismo do Brasil.'
""")

In [0]:
display(spark.sql("""
SELECT
    column_name,
    data_type,
    comment
FROM chegadas.information_schema.columns
WHERE table_schema = 'silver'
  AND table_name = 'brasil'
ORDER BY ordinal_position
"""))

In [0]:
display(spark.sql("""
SELECT
    COUNT(*) AS total_registros,
    MIN(ano) AS ano_inicial,
    MAX(ano) AS ano_final,
    COUNT(*) - COUNT(chegadas) AS chegadas_nulas
FROM chegadas.silver.brasil
"""))